<a href="https://colab.research.google.com/github/RajendarKumar/ai-agent_phase_1/blob/main/prompt_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "groq>=0.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Install pydantic for schema validation
!pip install --quiet pydantic

In [ ]:
from google.colab import userdata
key = userdata.get('GROQ_API_KEY')

In [ ]:
from groq import Groq
import json
import datetime
import re
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Literal
import pandas as pd



# Define Pydantic models for output validation
class EmailDetails(BaseModel):
    category: Literal["refund", "shipping", "complaint", "other"]
    order_number: Optional[str]
    amount: float # Assuming amount can be a float, based on 'number' type
    order_date: Optional[str] = Field(pattern="^\\d{4}-\\d{2}-\\d{2}$") # YYYY-MM-DD format

class RefundPolicyResult(BaseModel):
    refund_possible: Literal["YES", "NO", "NOT SURE"]
    reason: str

client = Groq(api_key=key)

def refund_rules_list(refund_policy:str):
  refund_rules_list = []
  # Split the string by rule number (e.g., '1. ' or '2. ')
  rules = re.split(r'\d+\.\s*', refund_policy)

  # Filter out empty strings that might result from the split and process each rule
  for i, rule_text in enumerate(rules):
      if rule_text.strip(): # Ensure the rule text is not empty
          rule_obj = {
              "rule": f"Rule {i}", # Assign a generic rule name, or infer from text if possible
              "value": rule_text.strip()
          }
          refund_rules_list.append(rule_obj)
  print(json.dumps(refund_rules_list, indent=2))
  return refund_rules_list

def extract_email_details(email_body: str, model: str = "openai/gpt-oss-120b"):
    """Extracts structured details from an email body using an LLM and validates against schema."""
    user_prompt = f"""
    Extract the following details from the email body into a JSON object:
    - category (refund, shipping, complaint, other)
    - order_number (string or null)
    - amount (number)
    - order_date (string, format YYYY-MM-DD)

    Email body: "{email_body}"

    JSON Schema:
    ```json
    {{
      "category": {{ "type": "string", "enum": ["refund", "shipping", "complaint", "other"] }},
      "order_number": {{ "type": ["string", "null"] }},
      "amount": {{ "type": "number" }},
      "order_date": {{ "type": ["string", "null"], "format": "YYYY-MM-DD" }}
    }}
    ```
    Ensure the output is valid JSON according to this schema.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0.0, # Make it deterministic for data extraction
        max_tokens=1024,
        top_p=1.0,
        stream=False,
        response_format={ "type": "json_object" }
    )
    print(f'email evaluation cost : {response.usage}')

    parsed_content = json.loads(response.choices[0].message.content)

    try:
        # Validate the parsed content against the Pydantic model
        validated_data = EmailDetails.model_validate(parsed_content)
        return validated_data.model_dump() # Return as dict
    except ValidationError as e:
        print(f"[ERROR] Email details validation failed: {e}")
        # In a real application, you might log this error and handle it gracefully
        # For now, we'll return the raw parsed content, but a robust system might raise it
        return parsed_content


def load_orders(file_name: str, file_ext: str):
    """Loads orders from a file (CSV or JSON) and returns them as a list of dictionaries."""
    orders_df = None
    if file_ext.lower() == '.csv':
        try:
            orders_df = pd.read_csv(file_name)
        except FileNotFoundError:
            print(f"Error: CSV file not found at {file_name}")
            return []
    elif file_ext.lower() == '.json':
        try:
            orders_df = pd.read_json(file_name)
        except FileNotFoundError:
            print(f"Error: JSON file not found at {file_name}")
            return []
    else:
        print(f"Error: Unsupported file extension: {file_ext}. Only .csv and .json are supported.")
        return []

    if orders_df is not None:
        return orders_df.to_dict(orient='records')
    return []
def verify_order(order: dict, order_id:str):
  if order is None:
    return False
  if order.get('order_id') == order_id:
            return True
  return False

def find_order_by_id(orders_list: list, order_id: str):
    """Finds an order in a list of order dictionaries by its order_id."""
    for order in orders_list:
        if order.get('order_id') == order_id:
            return order
    return None

def apply_refund_policy(extracted_data: dict, refund_policy: list, model: str = "openai/gpt-oss-120b", evaluation_date:datetime.date=None):
    """Applies the refund policy to extracted email data using an LLM and validates output."""
    extracted_data_json = json.dumps(extracted_data, indent=2)

    # Ensure evaluation_date is provided, default to today if not
    if evaluation_date is None:
        evaluation_date = datetime.date.today()
    evaluation_date_str = evaluation_date.strftime('%Y-%m-%d')

    user_prompt = f"""
    Given the following extracted email data, refund policy rules, and the current evaluation date,
    determine if a refund is possible. Provide 'YES', 'NO', or 'NOT SURE' for 'refund_possible'
    and a 'reason' explaining the decision.

    Extracted Email Data:
    ```json
    {extracted_data_json}
    ```

    Refund Policy rules:
    {refund_policy}

    Current Evaluation Date: {evaluation_date_str}

    JSON Schema for output:
    ```json
    {{
      "refund_possible": {{ "type": "string", "enum": ["YES", "NO", "NOT SURE"] }},
      "reason": {{ "type": "string" }}
    }}
    ```
    Ensure the output is valid JSON according to this schema.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0.0, # Make it deterministic for policy application
        max_tokens=1024,
        top_p=1.0,
        stream=False,
        response_format={ "type": "json_object" }
    )
    print(f'policy evaluation cost : {response.usage}')

    parsed_content = json.loads(response.choices[0].message.content)

    try:
        # Validate the parsed content against the Pydantic model
        validated_data = RefundPolicyResult.model_validate(parsed_content)
        return validated_data.model_dump() # Return as dict
    except ValidationError as e:
        print(f"[ERROR] Refund policy result validation failed: {e}")
        # In a real application, you might log this error and handle it gracefully
        # For now, we'll return the raw parsed content
        return parsed_content

def send_for_human_approval(data: dict):
    """Simulates sending refund data for human approval."""
    print("\n--- Reviewing the request ---")
    print(json.dumps(data, indent=2))
    # In a real system, this would involve sending an email, API call, etc.

## Module Imports and Data Models

This section outlines the essential Python modules imported for the refund processing system and defines the Pydantic models used for robust data validation.

### Imported Modules
*   **`groq`**: The client library for interacting with the Groq API, which provides the underlying Language Model (LLM) capabilities.
*   **`json`**: Standard Python library for working with JSON data, used for parsing LLM responses and formatting tool outputs.
*   **`datetime`**: Provides classes for working with dates and times, essential for handling `order_date` and `evaluation_date`.
*   **`re`**: The regular expression module, used for pattern matching, specifically for validating date formats.
*   **`pydantic`**: A powerful library for data validation and settings management, enabling clear and concise data models with type hints.
*   **`typing`**: Provides support for type hints, used in conjunction with Pydantic for defining flexible data structures.
*   **`pandas`**: A fundamental library for data manipulation and analysis, used here for loading order data from CSV/JSON files.

### Pydantic Data Models
Pydantic models are used to ensure that data extracted from emails and results from policy evaluations conform to predefined schemas. This helps maintain data integrity and prevents downstream errors.

*   **`EmailDetails(BaseModel)`**:
    *   **Purpose**: Validates and structures the information extracted from a customer's refund request email.
    *   **Fields**:
        *   `category` (`Literal["refund", "shipping", "complaint", "other"]`): The inferred category of the email.
        *   `order_number` (`Optional[str]`): The order identifier, if present.
        *   `amount` (`float`): The monetary amount associated with the order.
        *   `order_date` (`Optional[str]`): The date of the order, strictly enforced to 'YYYY-MM-DD' format using a `Field` pattern.

*   **`RefundPolicyResult(BaseModel)`**:
    *   **Purpose**: Validates and structures the outcome of the refund policy evaluation.
    *   **Fields**:
        *   `refund_possible` (`Literal["YES", "NO", "NOT SURE"]`): Indicates the eligibility for a refund based on the policy.
        *   `reason` (`str`): A textual explanation for the refund decision.

## Defining Tools for the Refund Processing Pipeline

To streamline the refund processing, we'll encapsulate the core functionalities into distinct tools. This makes the overall workflow more modular and easier to manage, especially when integrating with orchestration frameworks.

### Tool 1: `extract_order_from_email_body`

This tool is responsible for parsing unstructured email text and extracting relevant order details into a structured format. It utilizes the `extract_email_details` function, which leverages an LLM and Pydantic for robust extraction and validation.

### Tool 2: `lookup_order_in_database`

This tool simulates querying a database (in our case, a loaded CSV file) to retrieve a full order record based on an `order_id` extracted from the email. It combines the `load_orders` and `find_order_by_id` functions.

### Tool 3: `evaluate_refund_policy`

This tool applies the defined refund policy rules to the extracted order data and determines the eligibility for a refund, along with a reason. It uses the `apply_refund_policy` function, which employs an LLM to interpret and apply the rules dynamically.

In [ ]:
def extract_order_from_email_body_tool(email_body: str) -> dict:
    """Tool to extract structured order details from an email body.

    Args:
        email_body: The raw text content of the customer's email.

    Returns:
        A dictionary containing extracted details like category, order_number, amount, and order_date.
    """
    return extract_email_details(email_body)

def lookup_order_in_database_tool(order_id: str) -> Optional[dict]:
    """Tool to look up an order in the database (CSV file) by its order ID.

    Args:
        order_id: The ID of the order to look up.

    Returns:
        A dictionary representing the order if found, otherwise None.
    """
    # Assuming 'orders' is loaded globally or can be loaded here for the tool context
    # For simplicity, we'll reload it here, but in a production system, this might be optimized.
    orders = load_orders('/content/sample_data/order_data.csv', '.csv')
    return find_order_by_id(orders, order_id)

def evaluate_refund_policy_tool(order_details: dict, refund_policy_str: str, evaluation_date: Optional[str] = None) -> dict:
    """Tool to apply the refund policy to order details and get a refund decision.
    It expects 'order_details' to be a dictionary, preferably from a database lookup,
    containing a reliable 'order_date'.

    Args:
        order_details: A dictionary containing the extracted order information (ideally from database lookup).
                       Must contain 'order_date' for policy evaluation.
        refund_policy_str: A string describing the refund policy rules.
        evaluation_date: The date (YYYY-MM-DD) against which the policy should be evaluated. Defaults to today's date if not provided.

    Returns:
        A dictionary with 'refund_possible' (YES/NO/NOT SURE) and a 'reason'.
    """
    # Ensure order_details is a dictionary and contains 'order_date' for reliable policy evaluation
    if not isinstance(order_details, dict) or 'order_date' not in order_details:
        return {
            "refund_possible": "NOT SURE",
            "reason": "Order details are incomplete or missing 'order_date' for policy evaluation. Please ensure a valid order from the database is provided."
        }

    refund_rules = refund_rules_list(refund_policy_str)
    eval_date_obj = None
    if evaluation_date:
        try:
            eval_date_obj = datetime.datetime.strptime(evaluation_date, '%Y-%m-%d').date()
        except ValueError:
            print(f"Warning: Invalid date format for evaluation_date from LLM: {evaluation_date}. Falling back to today's date for policy evaluation.")
            eval_date_obj = datetime.date.today()
    return apply_refund_policy(order_details, refund_rules, evaluation_date=eval_date_obj)

def send_for_approval(data: dict):
    """Simulates sending data for approval.
    This tool is used to get the final approval from associate of the organization if the request is eligible the refund.
    Args:
        data: A dictionary containing the data to be sent for approval.
    """
    print("\n--- Sending for Approval ---")
    print("The following details require review:")
    print(json.dumps(data, indent=2))
    send_for_human_approval(data)
print("Tools `extract_order_from_email_body_tool`, `lookup_order_in_database_tool`, and `evaluate_refund_policy_tool` defined.")

Tools `extract_order_from_email_body_tool`, `lookup_order_in_database_tool`, and `evaluate_refund_policy_tool` defined.


In [ ]:
def tool_orchestrator(email_message: str, refund_policy_rules: str, evaluation_date: Optional[str] = None, model_name: str = "openai/gpt-oss-120b") -> str:
    """Orchestrates the refund processing using LLM and defined tools."""
    initial_user_content = f"""
Process the following refund request. First, use `extract_order_from_email_body_tool` to get details from the email. Then, use `lookup_order_in_database_tool` to verify the order using the extracted order number. Finally, use `evaluate_refund_policy_tool` with the verified order details from the database.

Email:
{email_message}

Refund Policy:
{refund_policy_rules}
"""

    if evaluation_date:
        initial_user_content += f" Use {evaluation_date} as the evaluation date for the policy."
    else:
        initial_user_content += " If no evaluation date is explicitly provided in the email or context, use today's date for policy evaluation."

    messages = [
        {"role": "user", "content": initial_user_content}
    ]

    tools = [
        {
            "type": "function",
            "function": {
                "name": "extract_order_from_email_body_tool",
                "description": "Extracts structured order details (category, order_number, amount, order_date) from a customer's email body. Always call this tool first to get initial order details.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "email_body": {"type": "string", "description": "The raw text content of the customer's email."},
                    },
                    "required": ["email_body"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "lookup_order_in_database_tool",
                "description": "Looks up a specific order in the system's database using its order ID. Call this after extracting the order_number from the email.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "order_id": {"type": "string", "description": "The unique identifier of the order to look up."},
                    },
                    "required": ["order_id"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "evaluate_refund_policy_tool",
                "description": "Applies the defined refund policy rules to order details and determines if a refund is possible, along with a reason. Call this after order details are available. The evaluation_date should be provided as 'YYYY-MM-DD' if known, otherwise it defaults to today's date.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "order_details": {"type": "object", "description": "A dictionary containing the extracted order information (e.g., order_id, amount, order_date)."},
                        "refund_policy_str": {"type": "string", "description": "A string describing the refund policy rules (e.g., '1. Orders older than 3 days are not eligible. 2. Amount cannot be more than 2500 Rupees.')."},
                        "evaluation_date": {"type": ["string", "null"], "format": "date", "description": "The date (YYYY-MM-DD) against which the policy should be evaluated. If null, today's date will be used."}
                    },
                    "required": ["order_details", "refund_policy_str"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "send_for_approval",
                "description": "Applies the defined refund approval tools if the request of the refund is eligible for the refund",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "data": {"type": "object", "description": "A dictionary containing the extracted order information (e.g., order_id, amount, order_date)."}
                    },
                    "required": ["data"],
                },
            },
        },
    ]

    available_tools = {
        "extract_order_from_email_body_tool": extract_order_from_email_body_tool,
        "lookup_order_in_database_tool": lookup_order_in_database_tool,
        "evaluate_refund_policy_tool": evaluate_refund_policy_tool,
        "send_for_approval": send_for_approval
    }

    MAX_ITERATIONS = 5 # Prevent infinite loops
    for i in range(MAX_ITERATIONS):
        print(f"\n--- LLM Turn {i+1} ---")
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.0,
            stream=False,
        )
        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            print(f"LLM decided to call tool(s):")
            for tool_call in message.tool_calls:
                function_name = tool_call.function.name
                function_to_call = available_tools[function_name]
                function_args = json.loads(tool_call.function.arguments)

                # Inject evaluation_date if specified in orchestrator call and not already in tool args
                # This is still useful if the LLM *didn't* provide an evaluation_date, but the orchestrator has one.
                # However, with the updated schema, the LLM should now be able to pass null correctly if it chooses.
                if function_name == 'evaluate_refund_policy_tool' and evaluation_date and 'evaluation_date' not in function_args:
                  function_args['evaluation_date'] = evaluation_date

                print(f"  Calling {function_name} with args: {json.dumps(function_args)}")
                function_response = function_to_call(**function_args)
                print(f"  Tool response: {json.dumps(function_response, indent=2)}")

                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": function_name,
                        "content": json.dumps(function_response),
                    }
                )

                # After evaluate_refund_policy_tool, if refund is possible or not sure, prompt LLM to call send_for_approval
                if function_name == 'evaluate_refund_policy_tool':
                    policy_result_obj = function_response
                    if policy_result_obj.get('refund_possible') in ['YES', 'NOT SURE']:
                        order_details_for_approval = function_args.get('order_details')
                        if order_details_for_approval:
                            messages.append(
                                {
                                    "role": "user",
                                    "content": f"The refund has been evaluated as {policy_result_obj.get('refund_possible')}. Please call the `send_for_approval` tool with these order details for final approval: {json.dumps(order_details_for_approval, indent=2)}"
                                }
                            )

        else:
            print("LLM final response (no more tool calls):")
            print(message.content)
            return message.content

    print("Orchestrator reached MAX_ITERATIONS without a final LLM response.")
    return "Error: Orchestration failed or timed out."


## `tool_orchestrator` Function Documentation

### Purpose
The `tool_orchestrator` function serves as the central control flow for processing customer refund requests. It leverages a Language Model (LLM) to intelligently select and execute a sequence of predefined tools to achieve a comprehensive refund evaluation, from initial email parsing to final approval.

### How it Works
1.  **Initial Prompt Construction**: It constructs an initial prompt for the LLM, instructing it on the overall task: extracting email details, looking up orders, and evaluating the refund policy. It also includes the customer's email message and the refund policy rules.
2.  **LLM Interaction with Tools**: The orchestrator enters a loop, interacting with the LLM. In each turn, it sends the current conversation history to the LLM. The LLM then decides whether to:
    *   Call one or more of the available tools (e.g., `extract_order_from_email_body_tool`, `lookup_order_in_database_tool`, `evaluate_refund_policy_tool`, `send_for_approval`).
    *   Provide a final textual response.
3.  **Tool Execution**: If the LLM decides to call a tool, the orchestrator executes that tool with the arguments provided by the LLM. The tool's response is then added back to the conversation history as a 'tool' message.
4.  **Iterative Process**: This process continues for a maximum number of iterations (`MAX_ITERATIONS`) or until the LLM provides a final response, indicating that the task is complete.
5.  **Dynamic `evaluation_date` Handling**: The `evaluation_date` parameter is dynamically passed to the `evaluate_refund_policy_tool` if specified, ensuring consistent policy application.
6.  **Conditional `send_for_approval` Call**: After the `evaluate_refund_policy_tool` is called, if the `refund_possible` result is 'YES' or 'NOT SURE', the orchestrator explicitly prompts the LLM to call the `send_for_approval` tool with the order details.

### Parameters
*   `email_message` (`str`): The full text content of the customer's refund request email.
*   `refund_policy_rules` (`str`): A string containing the refund policy rules that need to be applied (e.g., "1. Order not older than 3 days. 2. Amount not more than 2500 Rupees.").
*   `evaluation_date` (`Optional[str]`): An optional date string in 'YYYY-MM-DD' format to be used as the reference date for policy evaluation. If not provided, the `evaluate_refund_policy_tool` defaults to today's date.
*   `model_name` (`str`, default: "openai/gpt-oss-120b"): The name of the LLM model to be used for orchestrating tool calls and generating responses.

### Returns
*   `str`: The final response from the LLM, summarizing the outcome of the refund request processing.

### Available Tools
The orchestrator makes the following tools available to the LLM:
*   `extract_order_from_email_body_tool`: Parses an email to extract structured order details.
*   `lookup_order_in_database_tool`: Retrieves full order details from a simulated database using an order ID.
*   `evaluate_refund_policy_tool`: Applies the defined refund policy rules to order details to determine refund eligibility.
*   `send_for_approval`: Simulates sending eligible refund requests for human review and final approval.

In [ ]:
# --- Main execution (Updated to use tool_orchestrator) ---
Refund_policy = '1. The date of order should not be older than 3 days. 2. Amount of order can not be more than 2500 Rupees.'

base_email_body = "I have an order with id ORDS17807 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 20 Sept 2026"

print("\n--- Test Case 1 (Orchestrated): Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- ")
tool_orchestrator(
    email_message=base_email_body,
    refund_policy_rules=Refund_policy,
    evaluation_date="2026-09-23"
)



--- Test Case 1 (Orchestrated): Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- 

--- LLM Turn 1 ---
LLM decided to call tool(s):
  Calling extract_order_from_email_body_tool with args: {"email_body": "I have an order with id ORDS17807 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 20 Sept 2026"}
email evaluation cost : CompletionUsage(completion_tokens=227, prompt_tokens=304, total_tokens=531, completion_time=0.47553274, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=191), prompt_time=0.831844144, prompt_tokens_details=PromptTokensDetails(cached_tokens=256), queue_time=0.173253532, total_time=1.307376884)
  Tool response: {
  "category": "refund",
  "order_number": "ORDS17807",
  "amount": 30.0,
  "order_date": "2026-09-20"
}

--- LLM Turn 2 ---
LLM decided to call tool(s):
  Calling lookup_order_in_database_tool with args: {"order_id": "ORDS17807"}
  Tool response: {
  "order_id": "ORD

'**Refund Evaluation Result**\n\n- **Order ID:** ORDS17807  \n- **Order Date (from database):** 2026‑09‑24  \n- **Order Amount (from database):** 16,152.80\u202f₹  \n- **Evaluation Date:** 2026‑09‑23  \n\n**Policy Check**\n\n1. **Order age:** The order date is after the evaluation date, so the “not older than 3 days” rule is satisfied.  \n2. **Amount limit:** The order amount (16,152.80\u202f₹) exceeds the maximum allowed 2,500\u202f₹.\n\n**Conclusion:**  \nRefund **not possible**. The amount exceeds the policy limit (Rule\u202f2).  \n\nIf you have any further questions or need assistance with a different request, please let us know.'

In [ ]:

print("\n--- Test Case 2 (Orchestrated): Order Date: 2026-09-20, Evaluation Date: 2026-09-24 (4 days old) --- ")
tool_orchestrator(email_message=base_email_body, refund_policy_rules=Refund_policy, evaluation_date="2026-09-24")



--- Test Case 2 (Orchestrated): Order Date: 2026-09-20, Evaluation Date: 2026-09-24 (4 days old) --- 

--- LLM Turn 1 ---
LLM decided to call tool(s):
  Calling extract_order_from_email_body_tool with args: {"email_body": "I have an order with id ORDS17807 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 20 Sept 2026"}
email evaluation cost : CompletionUsage(completion_tokens=114, prompt_tokens=304, total_tokens=418, completion_time=0.238881111, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=78), prompt_time=0.014623099, prompt_tokens_details=None, queue_time=0.083970456, total_time=0.25350421)
  Tool response: {
  "category": "refund",
  "order_number": "ORDS17807",
  "amount": 30.0,
  "order_date": "2026-09-20"
}

--- LLM Turn 2 ---
LLM decided to call tool(s):
  Calling lookup_order_in_database_tool with args: {"order_id": "ORDS17807"}
  Tool response: {
  "order_id": "ORDS17807",
  "order_date": "2026-09-2

'**Refund Evaluation Result**\n\n- **Refund Possible:** No  \n- **Reason:** The order amount (₹16,152.80) exceeds the maximum allowed limit of ₹2,500 as per the refund policy (Rule\u202f2). Although the order date is within the 3‑day window (Rule\u202f1), the amount violation makes the refund ineligible.'

In [ ]:
print("\n--- Original Test Case (Orchestrated): Order Date 25 Sept 2026 (using system current date) ---")
email_body_3 = "I have an order with id 1232342344 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 25 Sept 2026"
tool_orchestrator(email_message=email_body_3, refund_policy_rules=Refund_policy, evaluation_date=datetime.date.today().strftime('%Y-%m-%d'))



--- Original Test Case (Orchestrated): Order Date 25 Sept 2026 (using system current date) ---

--- LLM Turn 1 ---
LLM decided to call tool(s):
  Calling extract_order_from_email_body_tool with args: {"email_body": "I have an order with id 1232342344 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 25 Sept 2026"}
email evaluation cost : CompletionUsage(completion_tokens=141, prompt_tokens=305, total_tokens=446, completion_time=0.291220971, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=105), prompt_time=0.01371157, prompt_tokens_details=None, queue_time=0.094818834, total_time=0.304932541)
  Tool response: {
  "category": "refund",
  "order_number": "1232342344",
  "amount": 30.0,
  "order_date": "2026-09-25"
}

--- LLM Turn 2 ---
LLM decided to call tool(s):
  Calling lookup_order_in_database_tool with args: {"order_id": "1232342344"}
  Tool response: null

--- LLM Turn 3 ---
LLM final response (no more tool 

'**Step\u202f1 – Extracted Order Details**\n\n| Field          | Value |\n|----------------|-------|\n| Order Number   | 1232342344 |\n| Amount (₹)     | 30 |\n| Order Date     | 2026‑09‑25 |\n\n**Step\u202f2 – Database Verification**\n\nThe order lookup (`lookup_order_in_database_tool`) returned **null**, meaning the order ID **1232342344** could not be found in our system.\n\n**Result**\n\nBecause the order cannot be verified in the database, we are unable to proceed with a refund evaluation or approval. Please double‑check the order number (it may be a typo or belong to a different system) and provide any additional reference (e.g., receipt, transaction ID) so we can locate the order and continue with the refund process.'

In [ ]:
print("\n--- Original Test Case (Orchestrated): Amount > 50, Order Date 20 Sept 2026 (using system current date) ---")
email_body_4 = "I have an order with id A1232342344 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2026"
tool_orchestrator(email_message=email_body_4, refund_policy_rules=Refund_policy, evaluation_date=datetime.date.today().strftime('%Y-%m-%d'))



--- Original Test Case (Orchestrated): Amount > 50, Order Date 20 Sept 2026 (using system current date) ---

--- LLM Turn 1 ---
LLM decided to call tool(s):
  Calling extract_order_from_email_body_tool with args: {"email_body": "I have an order with id A1232342344 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2026"}
email evaluation cost : CompletionUsage(completion_tokens=95, prompt_tokens=305, total_tokens=400, completion_time=0.199307991, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=58), prompt_time=0.076480089, prompt_tokens_details=None, queue_time=0.068379664, total_time=0.27578808)
  Tool response: {
  "category": "refund",
  "order_number": "A1232342344",
  "amount": 60.0,
  "order_date": "2026-09-20"
}

--- LLM Turn 2 ---
LLM decided to call tool(s):
  Calling lookup_order_in_database_tool with args: {"order_id": "A1232342344"}
  Tool response: null

--- LLM Turn 3 ---
LLM final response 

'**Step 1 – Extraction**  \n- Order ID: **A1232342344**  \n- Amount: **₹60**  \n- Order date: **2026‑09‑20**  \n\n**Step 2 – Database verification**  \nThe lookup for order **A1232342344** returned **no matching record** in our system.\n\n**Result**  \nBecause the order cannot be verified in the database, we are unable to proceed with a refund evaluation or approval. Please double‑check the order number (or provide additional details such as the email address used for the purchase) so we can locate the transaction and continue with the refund process.'

In [ ]:

print("\n--- Original Test Case (Orchestrated): Amount > 50, Order Date 20 Sept 2025 (very old, using system current date) ---")
email_body_5 = "I have an order with id ORDS11095 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2025"
tool_orchestrator(email_message=email_body_5, refund_policy_rules=Refund_policy, evaluation_date=datetime.date.today().strftime('%Y-%m-%d'))



--- Original Test Case (Orchestrated): Amount > 50, Order Date 20 Sept 2025 (very old, using system current date) ---

--- LLM Turn 1 ---
LLM decided to call tool(s):
  Calling extract_order_from_email_body_tool with args: {"email_body": "I have an order with id ORDS11095 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2025"}
email evaluation cost : CompletionUsage(completion_tokens=227, prompt_tokens=304, total_tokens=531, completion_time=0.474095879, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=191), prompt_time=0.012635957, prompt_tokens_details=None, queue_time=0.083689813, total_time=0.486731836)
  Tool response: {
  "category": "refund",
  "order_number": "ORDS11095",
  "amount": 60.0,
  "order_date": "2025-09-20"
}

--- LLM Turn 2 ---
LLM decided to call tool(s):
  Calling lookup_order_in_database_tool with args: {"order_id": "ORDS11095"}
  Tool response: {
  "order_id": "ORDS11095",
  "order

'The refund request has been forwarded for final approval. If you need any further assistance, feel free to let me know!'

In [ ]:
Refund_policy = '1. The date of order should not be older than 30 days. 2. Amount of order can not be more than 2500 Rupees.'
print("\n--- Test Case 1: Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- ")
tool_orchestrator(base_email_body, Refund_policy, evaluation_date="2026-09-23")



--- Test Case 1: Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- 


In [ ]:

base_email_body = "I have an order with id ORDS17807 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 20 Sept 2026"
print("\n--- Test Case 2: Order Date: 2026-09-20, Evaluation Date: 2026-09-24 (4 days old) --- ")
tool_orchestrator(base_email_body, Refund_policy, evaluation_date="2026-09-24")



--- Test Case 2: Order Date: 2026-09-20, Evaluation Date: 2026-09-24 (4 days old) --- 


In [ ]:
print("\n--- Original Test Case: Order Date 25 Sept 2026 (using system current date) ---")
email_body = "I have an order with id 1232342344 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 25 Sept 2026"
tool_orchestrator(email_body, Refund_policy, evaluation_date=datetime.date.today().strftime('%Y-%m-%d'))



--- Original Test Case: Order Date 25 Sept 2026 (using system current date) ---


In [ ]:
print("\n--- Original Test Case: Amount > 50, Order Date 20 Sept 2026 (using system current date) ---")
email_body = "I have an order with id A1232342344 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2026"
tool_orchestrator(email_body, Refund_policy, evaluation_date=datetime.date.today().strftime('%Y-%m-%d'))



--- Original Test Case: Amount > 50, Order Date 20 Sept 2026 (using system current date) ---


In [ ]:
print("\n--- Original Test Case: Amount > 50, Order Date 20 Sept 2025 (very old, using system current date) ---")
email_body = "I have an order with id ORDS11095 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2025"
tool_orchestrator(email_body, Refund_policy, evaluation_date=datetime.date.today().strftime('%Y-%m-%d'))



--- Original Test Case: Amount > 50, Order Date 20 Sept 2025 (very old, using system current date) ---


### Test Case for `send_for_approval`

This test case is designed to meet the refund policy criteria, ensuring that the `evaluate_refund_policy_tool` returns `YES` and subsequently prompts the LLM to call the `send_for_approval` tool.

In [ ]:
print("\n--- Test Case: Refund Possible, triggers send_for_approval ---")
email_body_for_approval = "I would like a refund for order ORDQ34225. The item was damaged. The amount was 664.6 rupees. The order date was 2026-09-25."
Refund_policy_for_test = '1. The date of order should not be older than 3 days. 2. Amount of order can not be more than 2500 Rupees.'

tool_orchestrator(
    email_message=email_body_for_approval,
    refund_policy_rules=Refund_policy_for_test,
    evaluation_date="2026-09-27" # 2 days after 2026-09-25, and amount 664.6 < 2500
)


--- Test Case: Refund Possible, triggers send_for_approval ---

--- LLM Turn 1 ---
LLM decided to call tool(s):
  Calling extract_order_from_email_body_tool with args: {"email_body": "I would like a refund for order ORDQ34225. The item was damaged. The amount was 664.6 rupees. The order date was 2026-09-25."}
email evaluation cost : CompletionUsage(completion_tokens=103, prompt_tokens=300, total_tokens=403, completion_time=0.216820894, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=65), prompt_time=0.013717891, prompt_tokens_details=None, queue_time=0.14252265, total_time=0.230538785)
  Tool response: {
  "category": "refund",
  "order_number": "ORDQ34225",
  "amount": 664.6,
  "order_date": "2026-09-25"
}

--- LLM Turn 2 ---
LLM decided to call tool(s):
  Calling lookup_order_in_database_tool with args: {"order_id": "ORDQ34225"}
  Tool response: {
  "order_id": "ORDQ34225",
  "order_date": "2026-09-25",
  "order_quantity": 2,
  "amount": 664.6,
  "rate": 332.3
}



'The refund request for **order ORDQ34225** has been processed:\n\n- **Order Date:** 2026‑09‑25  \n- **Amount:** ₹664.6  \n- **Evaluation Date:** 2026‑09‑27  \n\n**Policy Check:**  \n- The order is within the 3‑day window (2 days old).  \n- The amount is well below the ₹2,500 limit.  \n\n**Result:** The refund is eligible and has been forwarded for approval.'